# CyTOF Cell Lines — Per-cell Vendi Score & Marker Correlations

**Pipeline:**
1. Load 7 FCS files (breast cancer cell lines)
2. Rename metal-prefixed channels → clean marker names
3. Arcsinh(x/5) normalization — stored as the ingested layer
4. Ingest into a cytofstandard project (one run per cell line)
5. Compute per-cell Vendi score (k = 15, all biological markers)
6. Correlate Vendi score with each marker — combined heatmap across all lines

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats
import scipy.linalg
from pathlib import Path
from tqdm import tqdm
import fcsparser
import sys
# External packages needed for UMAP and PermCell
sys.path.append("/Users/ronguy/Dropbox/Work/CyTOF/HelperPackage/")
import mlx_umap  # noqa: F401
from PermCell_Smooth import *  # noqa: F401, F403

import cytofstandard
from cytofstandard import Project

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
plt.rcParams.update({
    'axes.labelsize': 12, 'axes.titlesize': 12,
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'pdf.fonttype': 42, 'ps.fonttype': 42,
})
sns.set_style("white")

## 1. File list, sample IDs, and channel mapping

In [ ]:
FCS_FILES = [
    '/Users/ronguy/Dropbox/Work/CyTOF/CyTOF_Christina/CyTOF_Data/CyTOF1_MDAMB468_Yael.fcs',
    '/Users/ronguy/Dropbox/Work/CyTOF/CyTOF_Christina/CyTOF_Data/CyTOF1_MCF7_Ori.fcs',
    '/Users/ronguy/Dropbox/Work/CyTOF/CyTOF_Christina/CyTOF_Data/CyTOF1_HCC1937.fcs',
    '/Users/ronguy/Dropbox/Work/CyTOF/CyTOF_Christina/CyTOF_Data/CyTOF1_SUM149.fcs',
    '/Users/ronguy/Dropbox/Work/CyTOF/CyTOF_Christina/CyTOF_Data/CyTOF1_MCF7_Yael.fcs',
    '/Users/ronguy/Dropbox/Work/CyTOF/CyTOF_Christina/CyTOF_Data/CyTOF1_HCC70.fcs',
    '/Users/ronguy/Dropbox/Work/CyTOF/CyTOF_Christina/CyTOF_Data/CyTOF1_MDAMB468_wo_CO2.fcs',
]

# Short run IDs derived from filenames
RUN_IDS = [
    Path(f).stem.replace('CyTOF1_', '').replace('_wo_CO2', '_woCO2')
    for f in FCS_FILES
]
print(RUN_IDS)

In [ ]:
# Metal-prefixed channel → clean marker name
CHANNEL_RENAME = {
    '140Ce_Cytokeratin_5': 'KRT5',
    '161Dy_H4K20me3':      'H4K20me3',
    '163Dy_ER':            'ER',
    '164Dy_CD49f':         'CD49f',
    '166Er_CD24':          'CD24',
    '167Er_GATA3':         'GATA3',
    '168Er_H3K27me3':      'H3K27me3',
    '170Er_H3K9me3':       'H3K9me3',
    '151Eu_H3K9me2':       'H3K9me2',
    '153Eu_H2AK119Ub':     'H2AK119ub',
    '155Gd_H3.3':          'H3.3',
    '156Gd_H3K64ac':       'H3K64ac',
    '158Gd_ZEB1':          'ZEB1',
    '160Gd_H3K27ac':       'H3K27ac',
    '165Ho_H3K36me3':      'H3K36me3',
    '115In_H3':            'H3',
    '175Lu_H3S28p':        'H3S28p',
    '142Nd_H3K27me2':      'H3K27me2',
    '143Nd_p53':           'p53',
    '144Nd_EZH2':          'EZH2',
    '145Nd_H3K4me3':       'H3K4me3',
    '150Nd_H3K4me1':       'H3K4me1',
    '141Pr_EpCAM':         'EpCAM',
    '147Sm_yH2A.X':        'pH2A.X',
    '149Sm_H3K36me2':      'H3K36me2',
    '152Sm_H4K16ac':       'H4K16ac',
    '154Sm_Vimentin':      'Vimentin',
    '159Tb_H4':            'H4',
    '169Tm_H3K9ac':        'H3K9ac',
    '171Yb_CD44':          'CD44',
    '172Yb_Ki-67':         'KI67',
    '174Yb_K8_K18':        'KRT8-18',
}

# Channels to drop (barcodes, DNA, live/dead, technical noise channels)
DROP_CHANNELS = {
    '191Ir_DNA1', '193Ir_DNA2', '195Pt_Live_Dead',
    '111Cd_Cd111_Barcode', '112Cd_Cd112_Barcode', '114Cd_Cd114_Barcode',
    '116Cd_Cd116_Barcode', '102Pd_Pd102_Barcode', '104Pd_Pd104_Barcode',
    '105Pd_Pd105_Barcode', '106Pd_Pd106_Cd106_Barcode', '108Pd_Pd108_Barcode',
    '110Pd_Pd110_Cd110_Barcode', '113In_Cd113_Barcode',
    'Center', 'Offset', 'Residual', 'Event_length', 'Time',
    '75As', '197Au', '190BCKG', '138Ba', '209Bi', '133Cs',
    '162Dy', '157Gd', '146Nd', '148Nd', '200Hg', '202Hg',
    '127I', '139La', '208Pb', '194Pt', '196Pt', '198Pt',
    '103Rh', '120Sn', '180Ta', '181Ta', '182W', '131Xe',
    '89Y', '173Yb', '176Yb',
}

# Final ordered marker list (biological, clean names)
BIO_MARKERS = list(CHANNEL_RENAME.values())
print(f"{len(BIO_MARKERS)} biological markers:", BIO_MARKERS)

## 2. Preprocessing: read FCS → rename → arcsinh(x/5) → save parquet

In [ ]:
COFACTOR = 5   # arcsinh cofactor applied post-ingestion
PREP_DIR = Path('/Users/ronguy/Dropbox/Work/CyTOF/Projects/CellLines_VendiScore/preprocessed')
PREP_DIR.mkdir(parents=True, exist_ok=True)

preprocessed = {}   # {run_id: Path to parquet}

for fcs_path, run_id in zip(FCS_FILES, RUN_IDS):
    out_path = PREP_DIR / f"{run_id}.parquet"
    preprocessed[run_id] = out_path

    if out_path.exists():
        print(f"  {run_id}: parquet already exists — skipping")
        continue

    _, df = fcsparser.parse(fcs_path, reformat_meta=True)

    # Keep only biological channels and rename — NO arcsinh here, raw counts preserved
    keep = [c for c in df.columns if c in CHANNEL_RENAME and c not in DROP_CHANNELS]
    df   = df[keep].rename(columns=CHANNEL_RENAME)

    df.to_parquet(out_path, index=False)
    print(f"  {run_id}: {df.shape[0]} cells × {df.shape[1]} markers → {out_path.name}")

## 3. Create cytofstandard project and ingest

In [ ]:
CYTOFSTD_DIR     = Path('/Users/ronguy/Dropbox/Work/CyTOF/Code/CyTOFSTD')
STANDARD_MARKERS = str(CYTOFSTD_DIR / 'cytof_marker_registry_files/standard_markers.csv')
MARKER_ALIASES   = str(CYTOFSTD_DIR / 'cytof_marker_registry_files/marker_aliases.yaml')
PROJECT_PATH     = '/Users/ronguy/Dropbox/Work/CyTOF/Projects/CellLines_VendiScore'

try:
    project = Project.load(PROJECT_PATH)
    print("Loaded existing project")
except Exception:
    # overwrite=True is safe here: the directory was pre-created by PREP_DIR.mkdir()
    # but contains no cytofstandard project files yet.
    project = Project.create(
        PROJECT_PATH,
        project_id='CellLines_VendiScore',
        project_name='Cell Lines Vendi Score',
        standard_marker_file=STANDARD_MARKERS,
        marker_alias_file=MARKER_ALIASES,
        overwrite=True,
    )
    print(f"Created new project at {PROJECT_PATH}")

In [ ]:
for run_id, parquet_path in preprocessed.items():
    if project.has_run(run_id):
        print(f"  {run_id}: already ingested — skipping")
        continue

    meta_path = PREP_DIR / f"{run_id}_meta.csv"
    pd.DataFrame([{
        "file_name": parquet_path.name,
        "sample_id": run_id,
        "line_id":   run_id,
    }]).to_csv(meta_path, index=False)

    run = project.add_run(run_id, run_name=run_id)
    run.ingest(
        files=[str(parquet_path)],
        sample_metadata=str(meta_path),
        strict_markers=False,
        allow_extra_markers=True,
    )
    adata = run.read_adata()
    print(f"  {run_id}: {adata.n_obs} cells × {adata.n_vars} markers")

project.list_runs()[['run_id', 'run_name', 'status']]

## 3b. Normalize with `run.normalize_with_cytof_transform`

Uses the external `cytof_transform` package to correct for technical variation using
core histone loading markers (`H3`, `H4`, `H3.3`) as controls, then stores the result
in `layers['normalized']` and sets `X` from that layer.

In [ ]:
# Histone loading controls — used to estimate the technical (DNA content) factor
CONTROL_MARKERS = [m for m in ['H3', 'H4', 'H3.3'] if m in ALL_MARKERS]

# All biological markers to correct (everything except the controls themselves)
MARKERS_TO_CORRECT = [m for m in ALL_MARKERS if m not in set(CONTROL_MARKERS)]

print(f"Control markers  : {CONTROL_MARKERS}")
print(f"Markers to correct: {MARKERS_TO_CORRECT}")

for run_id in tqdm(RUN_IDS, desc="Normalize"):
    run   = project.get_run(run_id)
    adata = run.read_adata()

    if "normalized" in adata.layers:
        print(f"  {run_id}: already normalized — skipping")
        run.set_x_from_layer("normalized")
        continue

    run.normalize_with_cytof_transform(
        control_markers=CONTROL_MARKERS,
        markers_to_correct=MARKERS_TO_CORRECT,
        source_layer="raw",
        corrected_layer="normalized",
        z_layer="normalized_z",
        arcsinh_cofactor=5.0,
        groupby_col="sample_id",   # one group per cell line
        inplace=True,
    )

    run.set_x_from_layer("normalized")
    print(f"  {run_id}: normalized, X set from 'normalized'")

## 4. Define marker subsets

Use all biological markers for Vendi (excluding H3, H3.3, H4 as pure loading controls).

In [ ]:
ref_adata   = project.get_run(RUN_IDS[0]).read_adata()
ALL_MARKERS = ref_adata.var_names.tolist()
print("Markers in zarr:", ALL_MARKERS)

# All markers for Vendi (exclude H3/H3.3/H4 loading controls)
MRK_VENDI = [m for m in ALL_MARKERS if m not in {'H3', 'H3.3', 'H4'}]

# Epigenetic and cell-identity subsets (for reference)
MRK_Epi = [m for m in [
    'H2AK119ub', 'H3K27ac', 'H3K27me2', 'H3K27me3',
    'H3K36me2', 'H3K36me3', 'H3K4me1', 'H3K4me3',
    'H3K64ac', 'H3K9ac', 'H3K9me2', 'H3K9me3',
    'H3S28p', 'H4K16ac', 'H4K20me3', 'pH2A.X',
] if m in ALL_MARKERS]

MRK_CI = [m for m in [
    'CD24', 'CD44', 'CD49f', 'EpCAM', 'GATA3', 'ER',
    'Vimentin', 'KRT5', 'KRT8-18', 'ZEB1', 'EZH2',
] if m in ALL_MARKERS]

print(f"\nMRK_VENDI ({len(MRK_VENDI)}): {MRK_VENDI}")
print(f"MRK_Epi   ({len(MRK_Epi)})")
print(f"MRK_CI    ({len(MRK_CI)})")

## 5. Per-cell Vendi score (k = 15)

Computed on `MRK_VENDI` (all biological markers except H3/H3.3/H4 loading controls).
Stored in `adata.obs["vendi_percell"]`.

> **Note:** These files have 50–90 k cells each. Each run takes ~30–60 s.

In [ ]:
K         = 15
N_BINS    = 10
OBS_KEY   = "vendi_percell"
MAX_CELLS = 10_000   # subsample per cell line for speed

for run_id in tqdm(RUN_IDS, desc="Per-cell Vendi"):
    run   = project.get_run(run_id)
    adata = run.read_adata()

    # Skip if already computed (checks the stored adata)
    # if OBS_KEY in adata.obs.columns:
    #     vs = adata.obs[OBS_KEY]
    #     print(f"  {run_id}: already computed ({adata.n_obs} cells) — skipping")
    #     continue

    # Subsample
    n   = adata.n_obs
    rng = np.random.default_rng(42)
    idx = rng.choice(n, min(MAX_CELLS, n), replace=False)

    if n > MAX_CELLS:
        print(f"  {run_id}: subsampling {n} → {len(idx)} cells")
        run._adata = adata[idx].copy()   # point the run at the subsample

    run.vendi_score(
        k=K,
        markers=MRK_Epi,
        n_bins=N_BINS,
        obs_key=OBS_KEY,
        metric="cosine",
        inplace=True,   # persists the (subsampled) adata with scores to zarr
    )

    adata = run.read_adata()
    vs    = adata.obs[OBS_KEY]
    print(f"  {run_id}: {adata.n_obs} cells  mean={vs.mean():.3f}  std={vs.std():.3f}")

In [ ]:
# Quick distribution check: violin per cell line
obs_list = []
for run_id in RUN_IDS:
    adata = project.get_run(run_id).read_adata()
    df = adata.obs[[OBS_KEY]].copy()
    df["cell_line"] = run_id
    obs_list.append(df)
obs_all = pd.concat(obs_list, ignore_index=True)

fig, ax = plt.subplots(figsize=(10, 4))
sns.violinplot(data=obs_all, x="cell_line", y=OBS_KEY, order=RUN_IDS,
               inner="box", linewidth=0.8, ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
ax.set_ylabel(f"Per-cell Vendi (k={K})")
ax.set_title("Per-cell Vendi score distribution per cell line")
plt.tight_layout()
plt.show()

In [ ]:
obs_all.groupby('cell_line').mean()

## 6. Pearson correlation: Vendi score × each marker, per cell line

For each cell line and each marker, compute Pearson r between the per-cell Vendi score
and that marker's arcsinh-transformed expression across all cells in the line.

Heatmap: markers (rows) × cell lines (cols), sorted by mean |r| across lines.

In [ ]:
corr_records = []   # [{run_id, marker, r, p}]

for run_id in tqdm(RUN_IDS, desc="Correlations"):
    run   = project.get_run(run_id)
    adata = run.read_adata()

    vs       = adata.obs[OBS_KEY].values.astype(float)
    var_names = adata.var_names.tolist()
    X        = np.asarray(adata.X, dtype=float)

    for mk in ALL_MARKERS:
        if mk not in var_names:
            continue
        j   = var_names.index(mk)
        expr = X[:, j]

        # Remove any nan/inf (rare but defensive)
        mask = np.isfinite(vs) & np.isfinite(expr)
        if mask.sum() < 10:
            continue

        r, p = scipy.stats.pearsonr(vs[mask], expr[mask])
        corr_records.append({"cell_line": run_id, "marker": mk, "r": r, "p": p})

corr_df = pd.DataFrame(corr_records)
print(corr_df.groupby("marker")["r"].describe().round(3))

In [ ]:
# Pivot to (marker × cell_line) matrix of Pearson r
heatmap_df = corr_df.pivot(index="marker", columns="cell_line", values="r")
heatmap_df = heatmap_df.reindex(columns=RUN_IDS)   # consistent column order

# Sort markers by mean absolute r across cell lines (most correlated at top)
mean_abs_r = heatmap_df.abs().mean(axis=1).sort_values(ascending=False)
heatmap_df = heatmap_df.loc[mean_abs_r.index]

print("Top 10 markers by mean |r|:")
print(mean_abs_r.head(10).round(3))

In [ ]:
# ── Combined heatmap: markers × cell lines ────────────────────────────────────
vmax = heatmap_df.abs().max().max()

fig, ax = plt.subplots(figsize=(len(RUN_IDS) * 1.2 + 2, len(ALL_MARKERS) * 0.38 + 1))

sns.heatmap(
    heatmap_df,
    ax=ax,
    cmap="RdBu_r",
    center=0,
    vmin=-vmax, vmax=vmax,
    annot=True, fmt=".2f", annot_kws={"size": 8},
    linewidths=0.3,
    cbar_kws={"label": "Pearson r  (Vendi score vs marker)", "shrink": 0.5},
)
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title(
    f"Correlation: per-cell Vendi score (k={K}) vs marker expression\n"
    f"arcsinh(x/{COFACTOR}) normalized — sorted by mean |r|",
    pad=10,
)
ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.savefig(
    Path('/Users/ronguy/Dropbox/Work/CyTOF/CyTOF_Christina') / 'Vendi_marker_correlations.pdf',
    dpi=200, bbox_inches='tight',
)
plt.show()

In [ ]:
# ── Scatter: top correlated marker vs Vendi score for each cell line ──────────
TOP_N = 4
top_markers = mean_abs_r.head(TOP_N).index.tolist()

fig, axes = plt.subplots(TOP_N, len(RUN_IDS),
                         figsize=(len(RUN_IDS) * 2.2, TOP_N * 2.2),
                         sharey="row")

for row, mk in enumerate(top_markers):
    for col, run_id in enumerate(RUN_IDS):
        ax = axes[row, col]
        run  = project.get_run(run_id)
        adata = run.read_adata()
        vs   = adata.obs[OBS_KEY].values
        j    = adata.var_names.tolist().index(mk)
        expr = np.asarray(adata.X[:, j]).ravel()

        # Subsample for plotting speed
        idx = np.random.default_rng(0).choice(len(vs), min(3000, len(vs)), replace=False)
        ax.scatter(expr[idx], vs[idx], s=1, alpha=0.3, rasterized=True)

        r = corr_df.query("cell_line == @run_id and marker == @mk")["r"].values
        if len(r):
            ax.set_title(f"r={r[0]:.2f}", fontsize=8, pad=2)

        if col == 0:
            ax.set_ylabel(mk, fontsize=9)
        if row == TOP_N - 1:
            ax.set_xlabel(run_id, fontsize=8, rotation=20, ha="right")
        ax.tick_params(labelsize=7)

fig.suptitle(f"Top {TOP_N} markers correlated with Vendi score", y=1.01)
plt.tight_layout()
plt.show()

## 7. Marker ablation — leave-one-out Vendi score

For each marker in `MRK_VENDI`, recompute the rarefied Vendi score for the whole
cell-line pool with that marker removed.

**delta = baseline − LOO score**
- Positive → removing the marker *decreases* diversity (marker drives heterogeneity)
- Negative → removing the marker *increases* diversity (marker was compressing the score)

Results are shown as a heatmap: markers × cell lines.

In [ ]:
M_RAREFY  = 500   # rarefaction depth (cells drawn per rep from the pool)
N_REPS_BL = 100   # reps for baseline
N_REPS_LO = 50    # reps for each LOO run

# ── Step 1: baseline Vendi per cell line (whole pool, MRK_VENDI) ──────────────
cache_bl = {}   # {run_id: vendi_score}

for run_id in tqdm(RUN_IDS, desc="Baseline"):
    run = project.get_run(run_id)
    df  = run.vendi_score(
        groupby="sample_id",
        markers=MRK_VENDI,
        n_bins=N_BINS,
        n_reps=N_REPS_BL,
        m=M_RAREFY,
        random_state=42,
        inplace=False,
    )
    cache_bl[run_id] = float(df["vendi_score"].iloc[0])
    print(f"  {run_id}: VS = {cache_bl[run_id]:.3f}")

print("\nBaseline Vendi scores:")
print(pd.Series(cache_bl).round(3))

In [ ]:
# ── Step 2: LOO — drop each marker and recompute ──────────────────────────────
loo_scores = {}   # {dropped_marker: {run_id: loo_score}}

for dropped in tqdm(MRK_VENDI, desc="LOO"):
    markers_loo = [mk for mk in MRK_VENDI if mk != dropped]
    loo_scores[dropped] = {}

    for run_id in RUN_IDS:
        run = project.get_run(run_id)
        df  = run.vendi_score(
            groupby="sample_id",
            markers=markers_loo,
            n_bins=N_BINS,
            n_reps=N_REPS_LO,
            m=M_RAREFY,
            random_state=42,
            inplace=False,
        )
        loo_scores[dropped][run_id] = float(df["vendi_score"].iloc[0])

# ── Step 3: delta matrix ──────────────────────────────────────────────────────
delta_data = {
    run_id: {
        dropped: cache_bl[run_id] - loo_scores[dropped][run_id]
        for dropped in MRK_VENDI
    }
    for run_id in RUN_IDS
}

delta_df = pd.DataFrame(delta_data).T   # (run_id × marker)
delta_df = delta_df.T                   # (marker × run_id)
delta_df = delta_df.reindex(columns=RUN_IDS)

# Sort markers by mean absolute delta
delta_df = delta_df.loc[delta_df.abs().mean(axis=1).sort_values(ascending=False).index]

print("Top markers by mean |delta|:")
print(delta_df.abs().mean(axis=1).head(10).round(3))

In [ ]:
# ── Heatmap: markers × cell lines ─────────────────────────────────────────────
vmax = delta_df.abs().max().max()

fig, ax = plt.subplots(figsize=(len(RUN_IDS) * 1.3 + 2, len(MRK_VENDI) * 0.38 + 1))
sns.heatmap(
    delta_df,
    ax=ax,
    cmap="RdBu_r", center=0, vmin=-vmax, vmax=vmax,
    annot=True, fmt=".2f", annot_kws={"size": 8},
    linewidths=0.3,
    cbar_kws={"label": "Δ Vendi  (baseline − LOO)", "shrink": 0.5},
)
ax.set_xlabel(""); ax.set_ylabel("")
ax.set_title("Marker ablation — Vendi score (whole pool)\n(+: drives diversity,  −: suppresses score)")
ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

# ── Bar chart: per-cell-line delta, sorted individually ───────────────────────
ncols = min(4, len(RUN_IDS))
nrows = -(-len(RUN_IDS) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 4 * nrows), sharey=False)
axes = np.atleast_1d(axes).ravel()

for ax, run_id in zip(axes, RUN_IDS):
    vals = delta_df[run_id].sort_values()
    ax.barh(vals.index, vals.values,
            color=["#d73027" if v > 0 else "#4575b4" for v in vals])
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(run_id, fontsize=10)
    ax.set_xlabel("Δ Vendi")
    ax.tick_params(axis="y", labelsize=8)

for ax in axes[len(RUN_IDS):]:
    ax.axis("off")

fig.suptitle("Per-cell-line marker ablation effect", y=1.01)
plt.tight_layout()
plt.show()

## 8. Marker contributions to individual eigenvectors

Squared loadings `v_j[k]²` of each marker on the dual-kernel eigenvectors,
and entropy-weighted contributions `(−λ_j log λ_j) × v_j[k]²` — per cell line.

In [ ]:
from sklearn.preprocessing import normalize as _normalize
import warnings as _w

# For each cell line: compute eigenvectors of the dual kernel on the whole pool
eig_data = {}   # {run_id: {'w': (d,), 'V2': (d,d)}}

for run_id in tqdm(RUN_IDS, desc="Eigenvectors"):
    run   = project.get_run(run_id)
    adata = run.read_adata()

    var_names = adata.var_names.tolist()
    col_idx   = [var_names.index(mk) for mk in MRK_VENDI if mk in var_names]
    X         = np.asarray(adata.X, dtype=np.float64)[:, col_idx]

    # Subsample for speed / stability
    rng = np.random.default_rng(42)
    idx = rng.choice(len(X), min(M_RAREFY, len(X)), replace=False)
    sub = X[idx]

    with _w.catch_warnings():
        _w.filterwarnings("ignore")
        from sklearn.preprocessing import KBinsDiscretizer
        binner = KBinsDiscretizer(n_bins=N_BINS, strategy="uniform", encode="ordinal")
        MM = binner.fit_transform(sub)

    MN = _normalize(MM, axis=1)
    S  = MN.T @ MN / len(MN)
    w, V = scipy.linalg.eigh(S)
    w = w[::-1]; V = V[:, ::-1]   # descending eigenvalue order

    eig_data[run_id] = {"w": w, "V2": V**2}

print("Done.")

In [ ]:
N_SHOW = 8   # top eigenvectors to display

# ── View 1: squared loadings heatmap, one panel per cell line ─────────────────
ncols = min(4, len(RUN_IDS))
nrows = -(-len(RUN_IDS) // ncols)
fig, axes = plt.subplots(nrows, ncols,
                         figsize=(5 * ncols, len(MRK_VENDI) * 0.32 + 1),
                         sharey=True)
axes = np.atleast_1d(axes).ravel()

for ax, run_id in zip(axes, RUN_IDS):
    V2 = eig_data[run_id]["V2"][:, :N_SHOW]
    w  = eig_data[run_id]["w"][:N_SHOW]
    df = pd.DataFrame(V2, index=MRK_VENDI,
                      columns=[f"EV{i+1}\nλ={w[i]:.3f}" for i in range(N_SHOW)])
    sns.heatmap(df, ax=ax, cmap="YlOrRd", vmin=0, vmax=1,
                annot=True, fmt=".2f", annot_kws={"size": 6},
                linewidths=0.3,
                cbar_kws={"label": "v²ⱼ[k]", "shrink": 0.4})
    ax.set_title(run_id, fontsize=9)
    ax.set_ylabel("Marker" if ax is axes[0] else "")
    ax.tick_params(axis="x", rotation=45, labelsize=7)

for ax in axes[len(RUN_IDS):]:
    ax.axis("off")

fig.suptitle(f"Squared loadings on dual-kernel eigenvectors (top {N_SHOW})", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── View 2: entropy-weighted loadings + total contribution summary ─────────────

# Build entropy-weighted DataFrames per cell line
ew_dfs = {}
for run_id in RUN_IDS:
    w  = eig_data[run_id]["w"][:N_SHOW]
    V2 = eig_data[run_id]["V2"][:, :N_SHOW]
    with np.errstate(divide="ignore", invalid="ignore"):
        hw = np.where(w > 0, -w * np.log(w), 0.0)
    ew = V2 * hw[np.newaxis, :]
    ew_dfs[run_id] = pd.DataFrame(
        ew, index=MRK_VENDI,
        columns=[f"EV{i+1}" for i in range(N_SHOW)],
    )

# Total entropy contribution per marker: sum over eigenvectors, then across cell lines
total_contrib = pd.DataFrame(
    {run_id: ew_dfs[run_id].sum(axis=1) for run_id in RUN_IDS}
)
total_contrib = total_contrib.loc[
    total_contrib.mean(axis=1).sort_values(ascending=False).index
]

fig, ax = plt.subplots(figsize=(len(RUN_IDS) * 1.3 + 2, len(MRK_VENDI) * 0.38 + 1))
sns.heatmap(
    total_contrib,
    ax=ax,
    cmap="YlOrRd",
    annot=True, fmt=".3f", annot_kws={"size": 8},
    linewidths=0.3,
    cbar_kws={"label": "Σⱼ (−λⱼ log λⱼ) · v²ⱼ[k]", "shrink": 0.5},
)
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title(
    f"Total entropy-weighted marker contribution (top {N_SHOW} eigenvectors)\n"
    "sorted by mean across cell lines"
)
ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()